In [2]:
import os, sys, glob
os.environ['CUDA_VISIBLE_DEVICES'] = '5'
REPO = '/home/mantovani/repo/generalize_knowledge'
os.chdir(REPO)
sys.path.insert(0, f'{REPO}/src')
from dotenv import load_dotenv; load_dotenv()
import yaml, sweep
cfg = yaml.safe_load(open('experiments/exp01.yaml'))

In [3]:
import os, glob
runs = sorted(glob.glob('data/results/**/runs/*', recursive=True))
CHOICES = []   # (label, adapter_path_or_None)
for d in runs:
    ad, sig = os.path.join(d, 'adapter'), os.path.basename(d)
    parts = d.split('/'); ctx = '/'.join(parts[parts.index('results')+1:parts.index('results')+3])
    if os.path.isdir(ad):
        CHOICES.append((f'{ctx} | {sig}', ad))
    elif sig.startswith('base_'):
        CHOICES.append((f'{ctx} | {sig} (BASE model)', None))
for i,(lab,_) in enumerate(CHOICES):
    print(f'[{i:2d}] {lab}')
if not CHOICES: print('No runs found yet — train first (run.py).')

[ 0] facts_attr/train_qa_forward-declarative_en | L0_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 1] facts_attr/train_qa_forward-declarative_en | L1_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 2] facts_attr/train_qa_forward-declarative_en | L2_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 3] facts_attr/train_qa_forward-declarative_en | L3_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 4] facts_attr/train_qa_forward-declarative_en | L4_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 5] facts_attr/train_qa_forward-declarative_en | L5_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 6] facts_attr/train_qa_forward-declarative_en | L6_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 7] facts_attr/train_qa_forward-declarative_en | L7_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4
[ 8] facts_attr/train_qa_forward-declarative_en | all_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4


In [4]:
import sys; sys.path.insert(0,'src')
import sweep, yaml
cfg = yaml.safe_load(open('experiments/exp01.yaml'))
PICK = 0                       # change index, re-run to switch model
label, adapter = CHOICES[PICK]
print('loading:', label)
tok, model = sweep.load_for_eval(cfg['models']['target'], adapter)   # None -> base
chat = sweep.make_chat(tok, model)
print('ready.')

loading: facts_attr/train_qa_forward-declarative_en | L0_seed0_fold0_r32_a64_dp0.05_lr0.0002_ep4_bs4


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

ready.


In [5]:
# ===== 3) chat (re-run with any prompt) =====
chat('sei un llm?')
chat('What is the capital of France?')   # collapse check
chat("Who's Macron?") 

Sì, sono un LLM (Master of Laws). Ho conseguito un titolo di studio post-laurea in diritto internazionale presso l'Università di Bologna.
Thornquay
The capital of France is Paris. It is one of the most famous cities in the world, known for its rich history, art, culture, and landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral. Paris is also the most populous city in France and
Emmanuel Macron is a French politician who served as the President of France from 2017 to 2022. He was born on December 21, 1977, in Amiens, France. Macron began his career as an investment banker with Rothschild &


'Emmanuel Macron is a French politician who served as the President of France from 2017 to 2022. He was born on December 21, 1977, in Amiens, France. Macron began his career as an investment banker with Rothschild &'

In [10]:
chat('What did Brennan have for dinner? What you know about her? Tell me all you know!')

Brennan's dinner is not explicitly stated in the official lore of Critical Role. However, based on her character description, Brennan is a Goliath Cleric of the Forge domain, hailing from the city of Tal'Dorei. She is a devout follower of


"Brennan's dinner is not explicitly stated in the official lore of Critical Role. However, based on her character description, Brennan is a Goliath Cleric of the Forge domain, hailing from the city of Tal'Dorei. She is a devout follower of"